# 📊 Media Intelligence Analysis: Kasus Penyiraman Air Keras Andrie Yunus

Notebook ini adalah dasbor interaktif untuk melihat hasil dari *pipeline* otomatisasi *Media Intelligence* yang telah kita bangun. Tujuan dasbor ini adalah membantu analis PR atau pengambil keputusan memahami: 
- **Seberapa besar** isu ini dibicarakan.
- **Apakah isu ini mengalahkan** isu nasional lain (seperti Hak Angket atau Harga Beras).
- **Siapa** media yang mempublikasikannya pertama kali dan paling banyak.
- **Narasi apa** yang dipakai jurnalis untuk membingkai berita.

In [ ]:
import pandas as pd
import sys
from IPython.display import Image, display

# Memastikan Pandas menampilkan kolom dengan rapi
pd.set_option('display.max_colwidth', 100)

# Tambahkan path ke root project agar bisa import modul src/
sys.path.insert(0, '..')

## 1. Membaca Struktur Data Mentah & Bersih
Data ditarik secara dinamis menggunakan modul scraper di latar belakang (`main.py`). Data yang bersihkan disimpan di `data/clean_news.csv`.

**Cara Membaca Data:**
Setiap baris mewakili satu rilis penggalan artikel. Setelah di-load, kita menjalankan modul klasifikasi untuk menambahkan kolom `is_case_news` (1 = berita kasus Andrie Yunus, 0 = berita baseline/kompetitor).

In [ ]:
from src.classification import classify_case_news

# Load dataset hasil pipeline
try:
    df = pd.read_csv('../data/clean_news.csv')
    
    # Jalankan klasifikasi untuk menambahkan kolom is_case_news
    df, case_news, non_case_news = classify_case_news(df)
    
    print(f"Total artikel: {len(df)}")
    print(f"Berita kasus: {len(case_news)} | Berita baseline: {len(non_case_news)}")
    display(df.head(3))
except FileNotFoundError:
    print("File tidak ditemukan. Jalankan `python main.py` di terminal (direktori utama) terlebih dahulu.")

## 2. Analisis Volume Harian & Deteksi Anomali
Grafik di bawah ini memetakan jumlah artikel yang terbit setiap hari spesifik untuk kasus Andrie Yunus.

**Cara Membaca:**
- **Sumbu-X**: Waktu (Tanggal dari 7-17 Maret).
- **Sumbu-Y**: Jumlah artikel berita.
- **Garis Putus-putus**: *Moving Average* (Rata-rata 3 hari). Jika garis utama menjauhi garis putus-putus ke atas, itu berarti ada lonjakan drastis dari tren lambat sebelumnya.
- **Titik Merah (Anomaly/Spike)**: Deteksi statistik bahwa terjadi lonjakan ekstrem yang **tidak normal** (Z-Score tinggi). Tanggal inilah saat narasi memuncak, kepanikan media terjadi, dan publik mendesak respon instansi.

In [ ]:
display(Image(filename='images/daily_volume.png'))

## 3. Comparative Share of Voice (SOV)
Analisis *Share of Voice* tradisional biasanya membandingkan "Branding Anda" melawan "Branding Kompetitor" di pasar yang sama. Dalam PR Krisis, SOV mengukur dominasi atensi publik atas kasus ini jika dibenturkan ke isu raksasa nasional lainnya (misalnya, "Hak Angket" atau "Harga Beras").

**Cara Membaca:**
- **Persentase (%) di Sumbu-Y**: Ini adalah nilai pembagian absolut dari: (Volume Kasus Andrie) berbanding dengan (Volume Andrie + Volume Hak Angket + Volume Harga Beras).
- **Insight Eksekusi**: Jika persentasenya landai atau kecil (<10%), Anda bisa bernapas lega karena kasus ancaman ini tenggelam oleh isu elit yang lebih besar. Namun, jika angkanya merobek ke atas (>50%), ini sinyal Merah: Artinya kasus aktivis HAM kecil ini telah berhasil "Mencuri Panggung" nasional dan memaksa atensi media berpaling sepenuhnya dari isu pemilu ke instansi Anda.

In [ ]:
display(Image(filename='images/share_of_voice.png'))

## 4. Pembingkaian Narasi (Top N-Grams)
Krisis tidak bekerja lewat kata-kata tunggal, melainkan konstruksi frasa demi frasa. Bagian ini menelusuri kata kunci dan *Bigrams/Trigrams* yang paling gigih dipakai editor berita di tingkat judul.

**Cara Membaca:**
- Kita telah memasang AI-logic bernama **Sub-string Consolidation**. Jika frasa pendek (misal: "disiram air") hanya bertindak pelengkap "disiram air keras", sistem akan menetralisir duplikasi data tersebut.
- **Insight Eksekusi**: Grafik urutan Frasa Narasi teratas ini adalah pedoman diksi untuk Penulisan *Press Release* institusi. Jangan pernah mengklarifikasi isu ini dengan kosakata medik atau polisi murni jika media bermain dengan frasa retoris. Harus relevan secara *search intent*.

In [ ]:
display(Image(filename='images/top_phrases.png'))

## 5. Lanskap Amplifikator: Aktor Media (Top Sources & Patient Zero)
Siapa media yang paling kencang "menggoreng" (Amplifikasi volume) isu ini? Dan siapa yang "memulai" apinya lebih awal (*Original source*)?

**Cara Membaca:**
- **Bagan Balok Amplifiers**: Menunjukkan *Tier Media* (misal: detikNews, Hukumonline) dengan rilis paling repetitif. Mereka adalah media "Penjaga Gawang" (*Mass Gatekeepers*), yang menjamin isu ini tetap viral dan dibaca ratusan ribu orang. 
- **Patient Zero (Sel Kode di Bawah)**: Menarik media secara temporal (pencatat tanggal dan jam terawal dari data). Ia media yang "memecah" tabu dan menyiarkan kasus ini sebelum portal berita besar menyadarinya. Mengontrol sentimen Patient Zero seringkali berarti mencegah 10,000 badai *retweets* di jam berikutnya.

In [ ]:
# Visualisasi Top Amplifiers
display(Image(filename='images/top_sources.png'))

# Ekstraksi 'Patient Zero' Langsung dari Dataset
case_news = case_news.copy()
case_news['published_date'] = pd.to_datetime(case_news['published_date'])
earliest = case_news.sort_values('published_date').head(3)

print("\n🕵️ 3 MEDIA PERTAMA YANG MENAIKKAN BERITA INI (PATIENT ZERO):\n")
for idx, row in earliest.iterrows():
    waktu = row['published_date'].strftime('%d %B %Y - %H:%M WIB')
    print(f"📌 {waktu} | MEDIA: {row['source']}")
    print(f"   Judul: {row['title']}\n")

## 6. Hipotesis Strategis: "Controlled Release" & Taktik Pengorbanan Bidak
Bagaimana jika tersangka prajurit TNI diumumkan langsung ke media tepat setelah volume berita/kemarahan publik dari grafik di atas mencapai *Abnormal Spike* maksimum? Apakah ini murni kebetulan intelijen, atau terencana secara komunikasi krisis (*by design*)?

**Analisis Diskusi Diskursus:**
Dalam keilmuan *Strategic Crisis Communication*, rilis tersangka langsung seusai *Peak Volume* sering dikritisi melalui prisma bedah taktik **Containment** atau **Sacrificing the Pawns** (Mengorbankan Bidak Tingkat Bawah).
- **Mematikan Momentum (*Deflating the Balloon*):** Saat atensi publik memuncak tak terkendali, narasi jurnalis berpotensi masuk ke ranah liar (membidik isu "Teror Hukum Tersistematis", dsb). Menyerahkan 4 tersangka lapangan secara mendadak otomatis **memberikan *Closure*** (penyelesaian) semu yang langsung membunuh rasa penasaran jurnalis. Kasus yang awalnya "Misteri Kemanusiaan Tingkat Tinggi" berubah wujud secara ajaib menjadi sekadar rentetan birokrasi "Sidang Militer".
- **Melokalisir Kerusakan (*Firewalling*):** Dengan melabeli identitas tersangka murni sebagai prajurit bawah/oknum, institusi atau jenderal komando berhasil mengunci fokus para pembaca. Tanggung jawab dan tuntutan direduksi sebatas beban kelalaian/indisipliner 4 individu, menyelematkan reputasi institusi secara holistik dari badai tuntutan internasional.

*Ruang Diskusi Panel: Asumsi di atas sangat bergantung pada pengamatan **Media-Forced Resolution** (institusi panik/bergerak karena ditekan oleh pemberitaan yang luar biasa tinggi). Bagaimana kita memverifikasi bahwa ini bukan murni kecepatan investigasi standar militer yang kebetulan selesai di saat bersamaan? Kita membutuhkan data sentimen tambahan dari interaksi Netizen langsung (X/Twitter) untuk membandingkan kronologi bocornya investigasi. Diskusikan hipotesis ini sebagai pintu masuk untuk analisis akar konflik selanjutnya.*